# Speed Testing OSKAR

Note all units of time are measured in seconds. Every other measurement is a simple amount e.g. number of sources or number of blocks.

## Initial Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

## Converting raw speeds to detailed info

In [2]:
# Load CSV
raw = pd.read_csv("speed_raw.csv")

raw

,x,y,z,dt,db,b
0,400,400,400,62751.015,21,150
1,10,10,100,58.824,15,39
2,400,400,1,744.814,2,3
3,100,100,100,2084.739,6,39


In [3]:
# Calulate box side length, channel area, area side length, and total source count
raw.insert(3, "n", raw.x * raw.y * raw.z)
raw.insert(3, "a", raw.x * raw.y)
raw.insert(3, "l", np.sqrt(raw.a))
raw.insert(3, "k", np.cbrt(raw.n))

# Calculate total elapsed time and time per block
raw.insert(9, "dt/db", raw.dt / raw.db)
raw.insert(11, "t", raw["dt/db"] * raw.b)

raw

,x,y,z,k,l,a,n,dt,db,dt/db,b,t
0,400,400,400,400.000000,400.0,160000,64000000,62751.015,21,2988.143571,150,448221.535714
1,10,10,100,21.544347,10.0,100,10000,58.824,15,3.921600,39,152.942400
2,400,400,1,54.288352,400.0,160000,160000,744.814,2,372.407000,3,1117.221000
3,100,100,100,100.000000,100.0,10000,1000000,2084.739,6,347.456500,39,13550.803500


In [4]:
# Save data to the calculation CSV
raw.to_csv("speed_all.csv", index=False)

## Calculate power law

The tight-bound time complexity should be dependent on three variables:

$t_{\max}=\Theta\left(g(x,y,z)\right); g(x,y,z) = \alpha {(xy)}^{\beta} z^{\gamma}: \alpha,\beta,\gamma\in\mathbb{R}^+$

Where $\alpha,\beta,\gamma$ are to be calculated using a formula fit. Since we expect $x=y$, we can simplify this to the following formula:

$g(x,y,z)\equiv g(a,z) = \alpha {a}^{\beta} z^{\gamma}, a = x\cdot y$

To make the fitting easier, we can plot the data on a log-log scale and use MLR to more easily evaluate the data:

$\log\left(g(a,z)\right) = f(a,z) = \log(\alpha) + \beta\log(a) + \gamma\log(z) \equiv F(A,Z) = \delta + \beta A + \gamma Z: \delta = \log(\alpha)$

In [5]:
# Load from the CSV
data = pd.read_csv("speed_all.csv")

In [6]:
# Get column data and convert to the log form
i = data[['a', 'z']]
I = np.log(data[['a', 'z']])
F = pd.Series(np.log(data.t))

print("x-axis values:")
print(I)
print("\ny-axis values:")
print(F)

x-axis values:
           a         z
0  11.982929  5.991465
1   4.605170  4.605170
2  11.982929  0.000000
3   9.210340  4.605170

y-axis values:
0    13.013043
1     5.030061
2     7.018600
3     9.514201
Name: t, dtype: float64


In [7]:
# Create the MLR model
model = LinearRegression()

model.fit(I, F)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2,)","[0.89,1.02]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['a','z']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-3.668
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,2


In [8]:
# Set variables corresponding to parameter values

delta = model.intercept_
[beta, gamma] = model.coef_
alpha = np.exp(delta)

print(alpha, beta, gamma)

0.02553200856918349 0.8949803733897755 1.0152485906089168


In [9]:
# Test the parameters on the same data

test_F = model.predict(I)
test_g = np.exp(test_F)

cdata = pd.DataFrame({"real" : data.t, "pred" : test_g})

print(cdata)

            real           pred
0  448221.535714  508637.019415
1     152.942400     168.866270
2    1117.221000    1160.567025
3   13550.803500   10411.268728


In [10]:
# Check relative error
print((cdata.real-cdata.pred)/cdata.real)

avg_error = np.mean(np.abs((cdata.real-cdata.pred)/cdata.real))

print(avg_error)

0   -0.134789
1   -0.104117
2   -0.038798
3    0.231686
dtype: float64
0.12734760825875616


In [ ]:
# Final formula
print("The final time complexity formula is:")
print("g(x,y,z) = %(alpha)5.4f * (x * y) ** %(beta)5.4f * z ** %(gamma)5.4f" % { 'alpha':alpha, 'beta':beta, 'gamma':gamma })

The final time complexity formula is:
g(x,y,z) = 0.0255 * (x * y) ** 0.8950 * z ** 1.0152


Or in $\LaTeX$ we have:

$\Theta\left(0.0255{(xy)}^{0.8950}{z}^{1.0152}\right)\approx\Theta\left(k^{2.8052}\right)\equiv\Theta\left(n^{0.9351}\right)$

Where $k$ is the average side length of a simulation box (with time complexity being exact), and $n$ is the number of sources in a box (with time complexity being averaged).

## Computer specifications

### General Information

| Info | Data |
|--|--|
| Model | NVIDIA GeForce RTX 2070 |
| Clock | 33 MHz |
| VRAM | 8 GiB |
| Max Power | 175 W |
| Compute Cap. | 7.5 |

### While OSKAR is Running in the Background

Assuming a `nice` value of 0.

| Info | Data |
|--|--|
| Performance State | P5 |
| VRAM Usage | 96 MiB |

(that's all I could collect in a reasonable amount of time)